# Transfer Learning Coding Lab  
## Happy vs. Sad Image Classifier with Google Colab

In this activity, you will build an image classifier using **transfer learning**.


### Big idea

Instead of training an image model from zero, we start with a model that already learned many visual patterns from many images.


## What is transfer learning?

Imagine a student already knows how to recognize basic visual patterns:

- edges,
- curves,
- colors,
- textures,
- shapes.

Instead of starting from nothing, we reuse that knowledge.

In this notebook:

1. We use a pre-trained image model called **MobileNetV2**.
2. We keep most of its learned visual knowledge.
3. We add a small new classifier for **Happy** vs. **Sad**.
4. We train only the new part using our images.

### Simple idea

**pre-trained model + small new classifier = transfer learning**

## Step 1: Upload your `happy_sad_dataset.zip` file

Before running the next cell:

1. download the zip file named `happy_sad_dataset.zip`.
2. Upload `happy_sad_dataset.zip` when Colab asks.

Expected structure:

```text
happy_sad_dataset.zip
    happy_sad_dataset/
        happy/
        sad/
```

In [ ]:
from google.colab import files
import zipfile
import os
import shutil
from pathlib import Path

uploaded = files.upload()

zip_name = None
for filename in uploaded.keys():
    if filename.endswith(".zip"):
        zip_name = filename
        break

if zip_name is None:
    raise ValueError("Please upload your happy_sad_dataset.zip file")

# Remove old data folder if it exists
data_root = Path("happy_sad_dataset")
if data_root.exists():
    shutil.rmtree(data_root)

# Unzip uploaded file
with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(".")

print("Uploaded and extracted:", zip_name)
print("Current files:", os.listdir("."))

## Step 2: Check your folders

This cell checks that `data/happy` and `data/sad` exist.

In [ ]:
from pathlib import Path

data_root = Path("happy_sad_dataset")
happy_dir = data_root / "happy"
sad_dir = data_root / "sad"

if not data_root.exists():
    raise FileNotFoundError("Cannot find folder named data. Make sure your zip contains happy_sad_dataset/happy and happy_sad_dataset/sad.")

if not happy_dir.exists():
    raise FileNotFoundError("Cannot find folder happy_sad_dataset/happy.")

if not sad_dir.exists():
    raise FileNotFoundError("Cannot find folder happy_sad_dataset/sad.")

happy_count = len([p for p in happy_dir.glob("*") if p.is_file()])
sad_count = len([p for p in sad_dir.glob("*") if p.is_file()])

print("Happy images:", happy_count)
print("Sad images:", sad_count)


## Step 3: Import tools

This cell loads the tools we need.

You do not need to memorize this code.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("Ready for transfer learning!")

## Step 4: Load images and automatically split the data

We only have one folder:

```text
happy_sad_dataset/happy
happy_sad_dataset/sad
```

So the notebook will automatically split the data:

- 80% for training
- 20% for validation

Training images help the model learn.  
Validation images help us check whether the model works on images it did not train on.

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 16
SEED = 123

train_dataset = tf.keras.utils.image_dataset_from_directory(
    data_root,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    data_root,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

class_names = train_dataset.class_names

print("Class names:", class_names)
print("Label 0 means:", class_names[0])
print("Label 1 means:", class_names[1])

## Step 5: Show sample images

Always look at your data before training a model.

Question:

> Do the Happy and Sad images look fair and balanced?

For example, if all happy images are bright and all sad images are dark, the model may learn **lighting**, not emotion.

In [ ]:
plt.figure(figsize=(10, 6))

for images, labels in train_dataset.take(1):
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label = int(labels[i].numpy()[0])
        plt.title(class_names[label])
        plt.axis("off")

plt.suptitle("Sample training images", fontsize=16)
plt.tight_layout()
plt.show()

# Part B: Build the transfer learning model

We will use a pre-trained model called **MobileNetV2**.

This model has already learned many useful visual patterns from many images.

We will use it as a **feature extractor**.

### What does feature extractor mean?

It means the pre-trained model looks at an image and finds useful clues, such as:

- edges,
- curves,
- textures,
- face parts,
- object shapes.

Then our small classifier decides:

**Happy or Sad?**

## Step 6: Prepare images for MobileNetV2

MobileNetV2 expects images to be scaled in a certain way.

This cell prepares the images for the pre-trained model.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

print("Datasets are ready.")

## Step 7: Load the pre-trained model

We use MobileNetV2 without its original final classifier.

Why?

Because its original job was not exactly Happy vs. Sad.  
We want to reuse its visual pattern knowledge and add our own small classifier.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze the base model so we do not change its learned visual knowledge
base_model.trainable = False

print("Pre-trained MobileNetV2 loaded.")
print("Number of layers in base model:", len(base_model.layers))

## Step 8: Add a small Happy/Sad classifier

Now we create a new model:

**input image → pre-trained model → small classifier → Happy/Sad prediction**

In [ ]:
inputs = tf.keras.Input(shape=(160, 160, 3))

# Prepare input for MobileNetV2
x = preprocess_input(inputs)

# Use the pre-trained model to extract features
x = base_model(x, training=False)

# Convert feature maps to a single feature vector
x = tf.keras.layers.GlobalAveragePooling2D()(x)

# Helps reduce overfitting
x = tf.keras.layers.Dropout(0.2)(x)

# Final output: one number between 0 and 1
# Close to 0 means class 0, close to 1 means class 1
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Step 9: Train the new classifier

This is transfer learning.

Most of the old model is frozen.  
We mainly train the new Happy/Sad classifier.

In [ ]:
EPOCHS = 5

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=validation_dataset
)

## Step 10: Plot training progress

This graph shows whether the model improved during training.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], marker="o", label="Training accuracy")
plt.plot(history.history["val_accuracy"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Transfer learning training progress")
plt.legend()
plt.grid(True)
plt.show()

## Step 11: Evaluate the model

Now we evaluate the model on the validation images.

In [ ]:
val_loss, val_accuracy = model.evaluate(validation_dataset)

print("Validation accuracy:", val_accuracy)
print("Validation accuracy as percentage:", round(val_accuracy * 100, 2), "%")

## Step 12: Show predictions on validation images

In [ ]:
plt.figure(figsize=(12, 8))

for images, labels in validation_dataset.take(1):
    predictions = model.predict(images, verbose=0)

    for i in range(min(9, len(images))):
        score = predictions[i][0]
        predicted_label = 1 if score >= 0.5 else 0
        true_label = int(labels[i].numpy()[0])

        correct = predicted_label == true_label
        result_word = "Correct" if correct else "Wrong"

        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(
            f"True: {class_names[true_label]}\nPred: {class_names[predicted_label]}\nScore: {score:.2f} ({result_word})"
        )
        plt.axis("off")

plt.tight_layout()
plt.show()

## Step 13: Upload one new image to test

You can upload a new happy or sad image and ask the model to predict it.

Try using:

- a photo,
- a drawing,
- an emoji face




In [ ]:
from google.colab import files
from tensorflow.keras.utils import load_img, img_to_array

uploaded = files.upload()

image_filename = list(uploaded.keys())[0]

# Load and resize image
img = load_img(image_filename, target_size=IMG_SIZE)
img_array = img_to_array(img)
img_batch = np.expand_dims(img_array, axis=0)

# Predict
score = model.predict(img_batch, verbose=0)[0][0]
predicted_label = 1 if score >= 0.5 else 0

plt.imshow(img)
plt.axis("off")
plt.title(f"Predicted: {class_names[predicted_label]} | Score: {score:.2f}")
plt.show()

print("Score:", score)
print("Predicted class:", class_names[predicted_label])
print("Reminder: AI can be wrong. Confidence does not guarantee correctness.")